# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaGaya/KGaya/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Two findings from the FlyRank research paper and my methodology questions:

Finding 1: The random forest model achieved roughly 3x improvement over
the hand-written baseline rule on Precision@50 (0.740 vs 0.240).

Methodology question: Where does the label come from? The label
is_declining_label is derived from trend_direction == "down", which is
calculated from the same 90-day window as the features. This means the
model is not predicting future decline — it is describing current state.
A stronger claim would require a prior feature window predicting a future
outcome window. The 3x improvement is real and directionally meaningful,
but it should be framed as "the model identifies currently declining pages
better than the rule" rather than "the model predicts which pages will decline."

Finding 2: The baseline rule Precision@50 was 0.240 — about 12 of the
top 50 flagged pages were actually declining.

Methodology question: Does the validation design support this? The paper
uses client-holdout validation, which is honest — pages from the same client
never appear in both train and test. However, the baseline rule does not use
a train/test split at all (it is a fixed formula), so comparing its Precision@50
to the model's Precision@50 on the holdout set is slightly asymmetric. The
baseline is scored on all data; the model is scored only on held-out clients.
This is a minor disclosure gap worth naming, not a flaw that invalidates the
finding.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Re-running my Week-5 Random Forest under an explicit client-grouped split
to show the before/after clearly.

Before: Week-5 model trained without explicit grouping check —
results may have included data leakage across clients.

After: Strict client-grouped split where no client appears in both
train and test. This is the honest number.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, average_precision_score

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                        REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = [
    'impressions_90d', 'sessions_90d', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position',
    'word_count', 'engagement_rate'
]

df_clean = df.dropna(subset=feature_cols).copy()

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.array(scores))
    topk = np.array(labels)[order[:k]]
    return topk.mean()

# =====================
# BEFORE — random split (no grouping)
# =====================
from sklearn.model_selection import train_test_split

X = df_clean[feature_cols]
y = df_clean['is_declining_label']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42)

rf_before = RandomForestClassifier(n_estimators=100, max_depth=6,
                                    class_weight='balanced', random_state=42)
rf_before.fit(X_train_r, y_train_r)
proba_before = rf_before.predict_proba(X_test_r)[:, 1]

p20_before = precision_at_k(proba_before, y_test_r, 20)
p50_before = precision_at_k(proba_before, y_test_r, 50)

print("BEFORE — Random split (no client grouping):")
print(f"  Precision@20: {p20_before:.3f}")
print(f"  Precision@50: {p50_before:.3f}")

# =====================
# AFTER — strict client-grouped split
# =====================
clients = df_clean['client_id'].unique()
np.random.seed(42)
np.random.shuffle(clients)
split = int(len(clients) * 0.8)
train_clients = set(clients[:split])
test_clients = set(clients[split:])

train = df_clean[df_clean['client_id'].isin(train_clients)]
test = df_clean[df_clean['client_id'].isin(test_clients)]

X_train_g = train[feature_cols]
y_train_g = train['is_declining_label']
X_test_g = test[feature_cols]
y_test_g = test['is_declining_label']

rf_after = RandomForestClassifier(n_estimators=100, max_depth=6,
                                   class_weight='balanced', random_state=42)
rf_after.fit(X_train_g, y_train_g)
proba_after = rf_after.predict_proba(X_test_g)[:, 1]

p20_after = precision_at_k(proba_after, y_test_g, 20)
p50_after = precision_at_k(proba_after, y_test_g, 50)

print("\nAFTER — Client-grouped split (honest):")
print(f"  Precision@20: {p20_after:.3f}")
print(f"  Precision@50: {p50_after:.3f}")

print("\nBefore/After comparison:")
print(f"  Precision@20: {p20_before:.3f} → {p20_after:.3f}")
print(f"  Precision@50: {p50_before:.3f} → {p50_after:.3f}")
print("\nNote: Any drop between before and after reflects the cost of honest validation.")
print("The grouped number is the one we trust.")


BEFORE — Random split (no client grouping):
  Precision@20: 0.950
  Precision@50: 0.880

AFTER — Client-grouped split (honest):
  Precision@20: 0.400
  Precision@50: 0.480

Before/After comparison:
  Precision@20: 0.950 → 0.400
  Precision@50: 0.880 → 0.480

Note: Any drop between before and after reflects the cost of honest validation.
The grouped number is the one we trust.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage audit on the final feature set.

Features used: impressions_90d, sessions_90d, content_age_days,
days_since_last_update, ctr, avg_position, word_count, engagement_rate.

Leakage check for each:
- impressions_90d: SAFE — observable before any review decision
- sessions_90d: SAFE — observable before any review decision  
- content_age_days: SAFE — knowable at decision time
- days_since_last_update: SAFE — knowable at decision time
- ctr: SAFE — calculated from impressions and clicks, both observable
- avg_position: SAFE — observable search ranking, known before decision
- word_count: SAFE — content metadata, knowable before decision
- engagement_rate: SAFE — observed metric from the same window

Label check:
- is_declining_label = trend_direction == "down" — used ONLY as the
  target variable, never as a feature. SAFE.
- trend_direction: excluded from features entirely. SAFE.
- trend_pct: not used. SAFE — this would be leakage since the label
  is derived from it.

Product flags check:
- health_score, priority_score, action_type: not in dataset by design.
  Cannot leak. SAFE.

Verdict: No leakage detected in the final feature set.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm no label-derived columns in feature set
label_derived = ['trend_direction', 'trend_pct', 'is_declining_label',
                  'health_score', 'priority_score', 'action_type']

print("Leakage audit — checking for forbidden columns in feature set:")
for col in label_derived:
    in_features = col in feature_cols
    in_data = col in df.columns
    status = "LEAK DETECTED" if in_features else "SAFE"
    print(f"  {col}: in features={in_features} | in dataset={in_data} | {status}")

print("\nAll features used:")
for f in feature_cols:
    print(f"  {f} — observable before decision point")

print("\nLeakage verdict: CLEAN")

# Correlation check — any feature suspiciously correlated with label?
corr = df_clean[feature_cols + ['is_declining_label']].corr()['is_declining_label'].drop('is_declining_label')
print("\nCorrelation of each feature with label (flag if > 0.7):")
print(corr.round(3).to_string())
high_corr = corr[abs(corr) > 0.7]
if len(high_corr) > 0:
    print(f"\nWARNING — suspiciously high correlation: {high_corr}")
else:
    print("\nNo suspiciously high correlations found. Leakage unlikely.")

Leakage audit — checking for forbidden columns in feature set:
  trend_direction: in features=False | in dataset=True | SAFE
  trend_pct: in features=False | in dataset=True | SAFE
  is_declining_label: in features=False | in dataset=True | SAFE
  health_score: in features=False | in dataset=False | SAFE
  priority_score: in features=False | in dataset=False | SAFE
  action_type: in features=False | in dataset=False | SAFE

All features used:
  impressions_90d — observable before decision point
  sessions_90d — observable before decision point
  content_age_days — observable before decision point
  days_since_last_update — observable before decision point
  ctr — observable before decision point
  avg_position — observable before decision point
  word_count — observable before decision point
  engagement_rate — observable before decision point

Leakage verdict: CLEAN

Correlation of each feature with label (flag if > 0.7):
impressions_90d          -0.023
sessions_90d             -0.025

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Claim rewrite — taking my boldest sentence and making it honest.

Original claim (too bold):
"The Random Forest model is 3x better than the baseline at identifying
pages that need review."

Why it goes too far:
- "Better" implies the model is correct — it is only more precise at
  the top of a ranked list on this specific dataset.
- "Need review" implies causation — the model observes patterns, it does
  not know what a page needs.
- The 3x figure comes from the starter pipeline, not my own validation run.

Rewritten in safe language:
"On this 30,000-row anonymized starter dataset, the Random Forest model
directionally outperformed the hand-written baseline rule at Precision@50,
suggesting that learned signal combinations may help prioritize content
review queues more effectively than a fixed rule. These results are
observational and decision-support only — the model identifies pages that
share characteristics with currently declining pages, and does not predict
future outcomes or guarantee that reviewing a flagged page will cause recovery."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the honest numbers that back the rewritten claim
print("Numbers behind the rewritten claim:")
print(f"Dataset: 30,000 rows, anonymized starter slice")
print(f"Validation: client-grouped holdout")
print(f"Baseline Precision@50 (Week 4 rule): measured on test split")
print(f"RF Precision@50 (this notebook, honest split): {p50_after:.3f}")
print(f"\nClaim scope:")
print("- Directional, not causal")
print("- Decision-support, not prediction")
print("- Observed on starter slice only, not the full warehouse")
print("- Does not claim refresh causes recovery")
print("- Does not claim Google algorithm factors identified")

Numbers behind the rewritten claim:
Dataset: 30,000 rows, anonymized starter slice
Validation: client-grouped holdout
Baseline Precision@50 (Week 4 rule): measured on test split
RF Precision@50 (this notebook, honest split): 0.480

Claim scope:
- Directional, not causal
- Decision-support, not prediction
- Observed on starter slice only, not the full warehouse
- Does not claim refresh causes recovery
- Does not claim Google algorithm factors identified


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.